# 🖐️ Thesis: Eksperymenty i Trening Klasyfikatora Gestów (Custom CNN + TensorBoard)

Notatnik badawczo-treningowy dla **Etapu 2** dwustopniowego potoku rozpoznawania gestów z drona:
1. **Etap 1:** Detekcja dłoni w kadrze (YOLO Nano).
2. **Etap 2 (Ten Notatnik):** Klasyfikacja wyciętego fragmentu dłoni (`crop`) do 1 z 8 gestów (`closehand`, `dislike`, `left`, `ok`, `openhand`, `right`, `shaka`, `v`).

### 🎯 Cele tego notatnika:
- **Eksperymentowanie z architekturą:** projektowanie i strojenie lekkiej sieci Custom CNN (optymalizowanej pod kątem urządzeń brzegowych / drona).
- **Śledzenie na żywo (TensorBoard):** wizualizacja krzywych uczenia (Loss/Accuracy) w czasie rzeczywistym bezpośrednio w notatniku.
- **Porównywanie modeli:** możliwość uruchomienia kilku wariantów (np. Custom CNN v1, Custom CNN v2, MobileNetV3) i nałożenia ich wykresów w TensorBoardzie.
- **Narzędzia badawcze do pracy inżynierskiej:** automatyczne generowanie Macierzy Pomyłek (Confusion Matrix), raportu klasyfikacji (Precision, Recall, F1) oraz wizualnej analizy błędów (Error Analysis).


## 1. ⚙️ Importy i konfiguracja środowiska obliczeniowego
Automatycznie wykrywamy najszybsze dostępne urządzenie:
- **Apple Silicon (MPS)** dla procesorów Apple M1/M2/M3/M4,
- **NVIDIA CUDA** dla procesorów graficznych (np. Google Colab),
- **CPU** jako uniwersalny fallback.


In [ ]:
import os
import sys
import time
import copy
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torch.utils.tensorboard import SummaryWriter

from sklearn.metrics import classification_report, confusion_matrix

# Wykrywanie optymalnego urządzenia
if torch.backends.mps.is_available():
    device = torch.device("mps")
    device_name = "Apple Silicon GPU (MPS)"
elif torch.cuda.is_available():
    device = torch.device("cuda")
    device_name = f"NVIDIA GPU: {torch.cuda.get_device_name(0)}"
else:
    device = torch.device("cpu")
    device_name = "CPU"

print(f"🚀 Aktywne urządzenie obliczeniowe: {device_name} ({device})")
print(f"📦 Wersja PyTorch: {torch.__version__}")


## 2. 📂 Przygotowanie danych i bezpieczne augmentacje

> ⚠️ **KLUCZOWA UWAGA DLA DATASETU:**  
> W zbiorze mamy gesty kierunkowe **`left`** oraz **`right`**!  
> **Nie wolno stosować `transforms.RandomHorizontalFlip()`**, ponieważ odbicie lustrzane zamienia gest „w lewo” na gest „w prawo”, co zafałszowałoby etykiety i rozregulowało sieć.  
> Stosujemy bezpieczne augmentacje odwzorowujące specyfikę widoku z drona:
> - Lekkie rotacje (przechyły drona $\pm 12^\circ$),
> - Skalowanie i przesunięcia (odległość od dłoni),
> - Zmiany jasności i nasycenia (`ColorJitter` – słońce/cień).


In [ ]:
# Ścieżka do datasetu (obsługuje uruchomienie w dane/notebooks oraz głównym folderze)
CWD = Path.cwd()
if (CWD / "data" / "cnn_dataset").exists():
    DATASET_DIR = CWD / "data" / "cnn_dataset"
elif (CWD.parent / "data" / "cnn_dataset").exists():
    DATASET_DIR = CWD.parent / "data" / "cnn_dataset"
else:
    DATASET_DIR = Path("/Users/michalskoczylas/Documents/studia/thesis/dane/data/cnn_dataset")

TRAIN_DIR = DATASET_DIR / "train"
VAL_DIR = DATASET_DIR / "val"

print(f"📁 Zbiór danych: {DATASET_DIR}")
assert TRAIN_DIR.exists() and VAL_DIR.exists(), f"Brak katalogu {TRAIN_DIR} lub {VAL_DIR}!"

# Hiperparametry przetwarzania obrazu
IMG_SIZE = 128
BATCH_SIZE = 32

# Transformacje dla zbioru treningowego (z augmentacją pod drona)
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(degrees=12),
    transforms.RandomAffine(degrees=0, translate=(0.06, 0.06), scale=(0.92, 1.08)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Transformacje dla zbioru walidacyjnego (czysty resize i normalizacja)
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Wczytanie zbiorów za pomocą PyTorch ImageFolder
train_dataset = datasets.ImageFolder(str(TRAIN_DIR), transform=train_transform)
val_dataset = datasets.ImageFolder(str(VAL_DIR), transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True if device.type != 'cpu' else False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

class_names = train_dataset.classes
num_classes = len(class_names)

print(f"✅ Klasy gestów ({num_classes}): {class_names}")
print(f"📸 Próbki treningowe : {len(train_dataset)}")
print(f"📸 Próbki walidacyjne: {len(val_dataset)}")


### 🖼️ Podgląd wylosowanych próbek treningowych po augmentacji
Upewnijmy się, że augmentacje są naturalne i nie zamazują konturów dłoni:


In [ ]:
def denormalize(tensor):
    """Odwraca normalizację ImageNet dla celów wizualizacji."""
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    inp = tensor.cpu().numpy().transpose((1, 2, 0))
    inp = std * inp + mean
    return np.clip(inp, 0, 1)

# Pobranie jednej paczki zdjęć
images, labels = next(iter(train_loader))

plt.figure(figsize=(14, 6))
for i in range(min(8, len(images))):
    ax = plt.subplot(2, 4, i + 1)
    img_disp = denormalize(images[i])
    plt.imshow(img_disp)
    plt.title(f"Klasa: {class_names[labels[i]]}", fontsize=11, fontweight="bold")
    plt.axis("off")
plt.tight_layout()
plt.show()


## 3. 📊 Uruchomienie TensorBoarda na żywo w Notebooku

TensorBoard pozwala obserwować na bieżąco:
- Spadek funkcji strat (**Loss**) na zbiorze treningowym i walidacyjnym,
- Wzrost dokładności (**Accuracy**),
- Wartość współczynnika uczenia (**Learning Rate**),
- Porównanie wielu eksperymentów na jednym wspólnym wykresie.


In [ ]:
# Załadowanie wbudowanego rozszerzenia TensorBoard w Jupyterze
%load_ext tensorboard

# Uruchomienie panelu nasłuchującego na katalog logów runs/gesture_cnn
%tensorboard --logdir runs/gesture_cnn


## 4. 🧠 Architektura: Lekka autorska sieć `GestureCNN`

Dla drona kluczowa jest niska liczba operacji zmiennoprzecinkowych (FLOPs) oraz mała liczba parametrów.  
Zaprojektowana architektura składa się z:
1. **4 bloków konwolucyjnych (ConvBlock):**
   - 2x warstwa Conv2D $3\times 3$ z `BatchNorm2d` i aktywacją `ReLU`,
   - `MaxPool2d(2, 2)` redukujący wymiar przestrzenny o połowę,
   - Lekki `Spatial Dropout (0.1)` zapobiegający ko-adaptacji cech.
2. **Global Average Pooling (`AdaptiveAvgPool2d(1, 1)`):**
   - Zastępuje tradycyjne, ciężkie spłaszczanie `Flatten()`.
   - Zmniejsza liczbę parametrów w części gęstej aż o **~80%**, co drastycznie redukuje overfitting na małym zbiorze danych!
3. **Kompaktowej głowicy klasyfikacyjnej:**
   - Warstwa Dense 128 z `Dropout(0.3)`,
   - Warstwa wyjściowa z $N$ neuronami (liczba gestów).


In [ ]:
class ConvBlock(nn.Module):
    """Pojedynczy podwójny blok konwolucyjny z normalizacją i regularyzacją."""
    def __init__(self, in_channels, out_channels, dropout_rate=0.1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(p=dropout_rate)
        )

    def forward(self, x):
        return self.conv(x)


class GestureCNN(nn.Module):
    """Customowa sieć CNN zoptymalizowana pod rozpoznawanie gestów na urządzeniach brzegowych."""
    def __init__(self, num_classes=8, in_channels=3, base_filters=32):
        super().__init__()
        
        # Ekstrakcja cech (Feature Extractor)
        self.features = nn.Sequential(
            ConvBlock(in_channels, base_filters),            # 128x128 -> 64x64
            ConvBlock(base_filters, base_filters * 2),       # 64x64   -> 32x32
            ConvBlock(base_filters * 2, base_filters * 4),   # 32x32   -> 16x16
            ConvBlock(base_filters * 4, base_filters * 8),   # 16x16   -> 8x8
        )
        
        # Global Average Pooling redukuje (batch, 256, 8, 8) -> (batch, 256, 1, 1)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        
        # Głowica klasyfikatora (Classifier Head)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=0.35),
            nn.Linear(base_filters * 8, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.25),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = self.classifier(x)
        return x

def print_model_summary(model, input_size=(1, 3, 128, 128)):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"📊 Liczba wszystkich parametrów : {total_params:,}")
    print(f"🔥 Parametry trenowalne          : {trainable_params:,}")
    print(f"💾 Szacowany rozmiar wag (FP32)  : {total_params * 4 / (1024 * 1024):.2f} MB")

# Test modelu na losowym wejściu
sample_model = GestureCNN(num_classes=num_classes)
print_model_summary(sample_model)


## 5. 🏋️ Uniwersalna pętla treningowa z rejestracją metryk do TensorBoarda

Funkcja `train_model()` dba o:
- Przeliczanie epoki uczącej i walidacyjnej,
- Logowanie metryk w czasie rzeczywistym do TensorBoarda (`writer.add_scalar`),
- Zapisywanie wykresu architektury do TensorBoarda (`writer.add_graph`),
- Dynamiczną regulację tempa uczenia (**Cosine Annealing Scheduler**),
- Zapisywanie najlepszych wag wg najniższego `Val Loss` / najwyższego `Val Acc`.


In [ ]:
def train_model(
    model, 
    train_loader, 
    val_loader, 
    num_epochs=45, 
    lr=0.001, 
    weight_decay=1e-4, 
    experiment_name="custom_cnn_v1",
    save_dir="../weights"
):
    save_path = Path(save_dir)
    save_path.mkdir(parents=True, exist_ok=True)
    best_weights_file = save_path / f"{experiment_name}_best.pt"
    
    # Inicjalizacja TensorBoard writera
    log_dir = Path("runs/gesture_cnn") / experiment_name
    writer = SummaryWriter(log_dir=str(log_dir))
    print(f"📈 Rozpoczynam eksperyment '{experiment_name}'. Logi: {log_dir}")
    
    # Przeniesienie modelu na akcelerator
    model = model.to(device)
    
    # Rejestracja grafu obliczeniowego w TensorBoardzie
    sample_batch, _ = next(iter(val_loader))
    writer.add_graph(model, sample_batch.to(device))
    
    # Funkcja straty z lekkim Label Smoothing (zapobiega nadmiernej pewności siebie modelu)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    
    # Optymalizator AdamW
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    # Scheduler łagodnie zmniejszający LR w rytmie cosinusa
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)
    
    best_val_acc = 0.0
    best_val_loss = float("inf")
    best_model_weights = copy.deepcopy(model.state_dict())
    
    start_time = time.time()
    
    print(f"\n{'Epoka':^7} | {'Train Loss':^10} | {'Train Acc':^10} | {'Val Loss':^10} | {'Val Acc':^10} | {'LR':^9}")
    print("-" * 70)
    
    for epoch in range(1, num_epochs + 1):
        # --- FAZA TRENINGU ---
        model.train()
        running_train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            
            running_train_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            train_correct += torch.sum(preds == targets.data).item()
            train_total += inputs.size(0)
            
        epoch_train_loss = running_train_loss / train_total
        epoch_train_acc = train_correct / train_total
        
        # --- FAZA WALIDACJI ---
        model.eval()
        running_val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                
                running_val_loss += loss.item() * inputs.size(0)
                _, preds = torch.max(outputs, 1)
                val_correct += torch.sum(preds == targets.data).item()
                val_total += inputs.size(0)
                
        epoch_val_loss = running_val_loss / val_total
        epoch_val_acc = val_correct / val_total
        
        current_lr = scheduler.get_last_lr()[0]
        scheduler.step()
        
        # --- LOGOWANIE DO TENSORBOARD ---
        writer.add_scalar("Loss/train", epoch_train_loss, epoch)
        writer.add_scalar("Loss/val", epoch_val_loss, epoch)
        writer.add_scalar("Accuracy/train", epoch_train_acc, epoch)
        writer.add_scalar("Accuracy/val", epoch_val_acc, epoch)
        writer.add_scalar("Learning_Rate", current_lr, epoch)
        
        # Wyświetlanie postępu
        marker = ""
        if epoch_val_acc > best_val_acc:
            best_val_acc = epoch_val_acc
            best_val_loss = epoch_val_loss
            best_model_weights = copy.deepcopy(model.state_dict())
            torch.save(best_model_weights, best_weights_file)
            marker = "⭐ Best Acc!"
            
        if epoch % 5 == 0 or epoch == num_epochs or marker:
            print(f"{epoch:^7} | {epoch_train_loss:^10.4f} | {epoch_train_acc*100:^9.2f}% | {epoch_val_loss:^10.4f} | {epoch_val_acc*100:^9.2f}% | {current_lr:^9.6f} {marker}")
            
    time_elapsed = time.time() - start_time
    print("-" * 70)
    print(f"🏁 Trening ukończony w: {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s")
    print(f"🏆 Najwyższy Val Accuracy: {best_val_acc * 100:.2f}% (zapisano w {best_weights_file})")
    
    writer.close()
    
    # Przywracamy najlepsze wagi
    model.load_state_dict(best_model_weights)
    return model


## 6. 🚀 Eksperyment 1: Trening modelu `GestureCNN` (Custom CNN)

Uruchamiamy 40 epok. Zauważ, jak w okienku TensorBoard powyżej krzywe na bieżąco się aktualizują (możesz też kliknąć ikonkę odświeżania w TensorBoardzie).


In [ ]:
# Inicjalizacja i uruchomienie treningu
model_custom = GestureCNN(num_classes=num_classes, base_filters=32)

# Katalog zapisu wag
WEIGHTS_DIR = DATASET_DIR.parent.parent / "weights"

trained_custom_cnn = train_model(
    model=model_custom,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=40,
    lr=0.001,
    weight_decay=1e-4,
    experiment_name="custom_cnn_v1",
    save_dir=str(WEIGHTS_DIR)
)


## 7. 🎯 Ewaluacja i metryki do pracy inżynierskiej

Dokładna analiza wyników na zbiorze walidacyjnym:
1. **Macierz pomyłek (Confusion Matrix)** – precyzyjnie pokazuje, które gesty model rozpoznaje bezbłędnie, a które bywają mylone.
2. **Raport klasyfikacji (Precision, Recall, F1-Score)** – metryki do zamieszczenia w tabeli wyników pracy dyplomowej.


In [ ]:
def evaluate_model(model, val_loader, class_names, title="Confusion Matrix"):
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(targets.numpy())
            
    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)
    
    # 1. Raport klasyfikacji
    print("=" * 60)
    print("📋 SZCZEGÓŁOWY RAPORT KLASYFIKACJI:")
    print("=" * 60)
    print(classification_report(all_targets, all_preds, target_names=class_names, digits=4))
    
    # 2. Macierz pomyłek (Confusion Matrix)
    cm = confusion_matrix(all_targets, all_preds)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(
        cm_norm, 
        annot=cm, 
        fmt='d', 
        cmap='Blues', 
        xticklabels=class_names, 
        yticklabels=class_names,
        cbar_kws={'label': 'Prawdopodobieństwo (znormalizowane)'}
    )
    plt.title(f"{title} (zbiór walidacyjny)", fontsize=14, pad=15)
    plt.ylabel('Rzeczywista klasa (Ground Truth)', fontsize=12)
    plt.xlabel('Predykcja modelu', fontsize=12)
    plt.xticks(rotation=45)
    plt.tight_layout()
    
    # Zapis do pliku graficznego (np. pod LaTeX / rozdział pracy)
    output_cm_path = DATASET_DIR.parent / "reports"
    output_cm_path.mkdir(exist_ok=True)
    plt.savefig(output_cm_path / "confusion_matrix_custom_cnn.png", dpi=300)
    plt.show()
    
    print(f"💾 Zapisano wykres macierzy pomyłek: {output_cm_path / 'confusion_matrix_custom_cnn.png'}")
    return all_preds, all_targets

preds, targets = evaluate_model(trained_custom_cnn, val_loader, class_names, title="Custom CNN - Macierz Pomyłek")


### 🔍 Jakościowa analiza błędów (Error Analysis)
Przejrzyjmy wycinki, na których model popełnił błąd, aby zrozumieć przyczynę (np. nietypowy kąt, słabe oświetlenie czy rozmycie ruchu):


In [ ]:
def show_misclassified_samples(model, val_loader, class_names, max_images=8):
    model.eval()
    errors = []
    
    with torch.no_grad():
        for inputs, targets in val_loader:
            outputs = model(inputs.to(device))
            probs = torch.softmax(outputs, dim=1)
            confidences, preds = torch.max(probs, 1)
            
            for i in range(len(targets)):
                if preds[i].cpu() != targets[i]:
                    errors.append({
                        'image': inputs[i],
                        'true_label': class_names[targets[i]],
                        'pred_label': class_names[preds[i].cpu()],
                        'confidence': confidences[i].item() * 100
                    })
                if len(errors) >= max_images:
                    break
            if len(errors) >= max_images:
                break
                
    if not errors:
        print("🎉 Model nie popełnił żadnego błędu na wybranym podzbiorze!")
        return
        
    cols = min(4, len(errors))
    rows = (len(errors) + cols - 1) // cols
    plt.figure(figsize=(4 * cols, 4 * rows))
    
    for idx, err in enumerate(errors):
        plt.subplot(rows, cols, idx + 1)
        plt.imshow(denormalize(err['image']))
        plt.title(
            f"True: {err['true_label']}\nPred: {err['pred_label']} ({err['confidence']:.1f}%)",
            color="darkred", 
            fontweight="bold",
            fontsize=11
        )
        plt.axis("off")
        
    plt.suptitle("Błędnie zaklasyfikowane próbki walidacyjne (Error Analysis)", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

show_misclassified_samples(trained_custom_cnn, val_loader, class_names)


## 8. 🔬 Eksperyment 2 (Porównanie do pracy): Transfer Learning z MobileNetV3-Small

Jedną z największych zalet TensorBoarda jest **porównywanie eksperymentów na wspólnych wykresach**.  
Wytrenujmy teraz model bazowy (**MobileNetV3-Small** ze zmienioną głowicą na 8 klas) i zobaczmy oba modele zestawione w TensorBoardzie!


In [ ]:
# Inicjalizacja MobileNetV3-Small z wagami pretrenowanymi na ImageNet
mobilenet_model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)

# Zamiana ostatniej warstwy liniowej na 8 klas naszego zadania
in_features = mobilenet_model.classifier[3].in_features
mobilenet_model.classifier[3] = nn.Linear(in_features, num_classes)

print("MobileNetV3-Small załadowany!")
print_model_summary(mobilenet_model)

# Trening z rejestracją do 'mobilenet_v3_baseline' w runs/gesture_cnn
trained_mobilenet = train_model(
    model=mobilenet_model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=35,
    lr=0.0005,
    weight_decay=1e-4,
    experiment_name="mobilenet_v3_baseline",
    save_dir=str(WEIGHTS_DIR)
)


## 9. 📦 Eksport modelu produkcyjnego pod drona (ONNX & TorchScript)

Na koniec eksportujemy najlepszy model do formatu **ONNX** oraz **TorchScript**, co umożliwi błyskawiczną inferencję w C++ / Pythonie na komputerze pokładowym drona bez narzutu frameworka trenującego.


In [ ]:
def export_model_for_deployment(model, output_dir="../weights", input_size=(1, 3, 128, 128)):
    model.eval()
    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    
    dummy_input = torch.randn(*input_size).to(device)
    
    # 1. Eksport TorchScript
    ts_file = out_path / "gesture_cnn.torchscript.pt"
    traced_model = torch.jit.trace(model, dummy_input)
    traced_model.save(str(ts_file))
    print(f"✅ Zapisano TorchScript: {ts_file} ({ts_file.stat().st_size / (1024*1024):.2f} MB)")
    
    # 2. Eksport ONNX (uniwersalny format pod ONNX Runtime / TensorRT / OpenVINO)
    onnx_file = out_path / "gesture_cnn.onnx"
    torch.onnx.export(
        model,
        dummy_input,
        str(onnx_file),
        export_params=True,
        opset_version=14,
        do_constant_folding=True,
        input_names=['input_crop'],
        output_names=['gesture_logits'],
        dynamic_axes={'input_crop': {0: 'batch_size'}, 'gesture_logits': {0: 'batch_size'}}
    )
    print(f"✅ Zapisano ONNX       : {onnx_file} ({onnx_file.stat().st_size / (1024*1024):.2f} MB)")

export_model_for_deployment(trained_custom_cnn, output_dir=str(WEIGHTS_DIR))
